[![colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ws2026ies/exercises/blob/main/03_2_Layer_Atmosphere.ipynb)

# Exercise: Sensitivity Analysis of a Two-Layer Atmosphere Model

Learning objectives of this exercise:
- Get a basic idea of the physics behind the temperature profiles of the atmosphere
- Get familiar with a simple atmospheric model, understand model parameters and model sensitivity
- Modify your code, such that it considers also climate change.

---

## Introduction

Understanding the impact of key parameters on climate models is crucial for gaining insights into Earth's energy balance. In this exercise, we will conduct a sensitivity analysis on a simplified two-layer atmosphere model implemented in Python, the original R code, visualisations and physical background are taken from this website: https://simpleserioussolvable.shinyapps.io/2layer/. The model calculates the planetary energy budget, considering the influence of emissivity and albedo on the temperature distribution and outgoing longwave radiation.


## Planetary Energy Balance Model

![2-layer-atmosphere](https://simpleserioussolvable.shinyapps.io/2layer/_w_c8ba5b7c053f4aa2be994948f59f8a8b/model.png)

*Image from https://simpleserioussolvable.shinyapps.io/2layer/

## Variables

1. $T_S$, $T_1$, and $T_2$: Temperatures of the surface, the lower layer, and the upper layer, respectively.

2. $\epsilon_1$ and $\epsilon_2$: Emissivities of the lower layer and the upper layer, respectively.

3. $H_S$ and $H_L$: Convective heat fluxes from the surface to the lower layer and from the lower layer to the upper layer, respectively.

4. $\alpha$: Planet albedo, representing the fraction of incoming solar radiation that is reflected back into space.

5. $S_0$: Solar constant.

The absorbed solar radiation is

$$
S = \frac{S_0}{4}(1-\alpha)
$$

### 1. Surface energy balance

$$
S + \epsilon_1 \sigma T_1^4
+ (1-\epsilon_1)\epsilon_2 \sigma T_2^4
= \sigma T_s^4 + H_s
$$

### 2. Lower atmosphere energy balance

$$
\epsilon_1 \sigma T_s^4
+ \epsilon_1\epsilon_2 \sigma T_2^4
+ H_s
= 2\epsilon_1 \sigma T_1^4 + H_L
$$

### 3. Upper atmosphere energy balance

$$
(1-\epsilon_1)\epsilon_2 \sigma T_s^4
+ \epsilon_1\epsilon_2 \sigma T_1^4
+ H_L
= 2\epsilon_2 \sigma T_2^4
$$




### Matrix form of the energy balance equations

The three energy balance equations can be expressed as:

$$
\boxed{\mathbf{A}\mathbf{x}=\mathbf{b}}
$$

where the transfer matrix is:

$$
\mathbf{A} =
\begin{bmatrix}
-1 & \epsilon_1 & (1-\epsilon_1)\epsilon_2 \\
\epsilon_1 & -2\epsilon_1 & \epsilon_1\epsilon_2 \\
(1-\epsilon_1)\epsilon_2 & \epsilon_1\epsilon_2 & -2\epsilon_2
\end{bmatrix}
$$

The unknown temperature vector is:

$$
\mathbf{x} =
\begin{bmatrix}
T_s^4 \\
T_1^4 \\
T_2^4
\end{bmatrix}
$$

And the right-hand-side vector is:

$$
\mathbf{b} =
\frac{1}{\sigma}
\begin{bmatrix}
H_s-S \\
H_L-H_s \\
-H_L
\end{bmatrix}
$$

Therefore, the complete system is:

$$
\begin{bmatrix}
-1 & \epsilon_1 & (1-\epsilon_1)\epsilon_2 \\
\epsilon_1 & -2\epsilon_1 & \epsilon_1\epsilon_2 \\
(1-\epsilon_1)\epsilon_2 & \epsilon_1\epsilon_2 & -2\epsilon_2
\end{bmatrix}
\begin{bmatrix}
T_s^4 \\
T_1^4 \\
T_2^4
\end{bmatrix}
=
\frac{1}{\sigma}
\begin{bmatrix}
H_s-S \\
H_L-H_s \\
-H_L
\end{bmatrix}
$$



### Solving for temperatures

The matrix equation is

$$
\mathbf{A}\mathbf{x}=\mathbf{b}
$$

Assuming $\mathbf{A}$ is invertible, the formal solution is:

$$
\mathbf{x}=\mathbf{A}^{-1}\mathbf{b}
$$

Thus,

$$
\begin{bmatrix}
T_s^4 \\
T_1^4 \\
T_2^4
\end{bmatrix}
=
\mathbf{A}^{-1}\mathbf{b}
$$

The temperatures are obtained by taking the fourth root:

$$
T_s = (x_1)^{1/4}
$$

$$
T_1 = (x_2)^{1/4}
$$

$$
T_2 = (x_3)^{1/4}
$$



### Outgoing longwave radiation

The total outgoing longwave radiation at the top of the atmosphere is:

$$
OLR =
\sigma T_s^4(1-\epsilon_1)(1-\epsilon_2)
+ \epsilon_1\sigma T_1^4(1-\epsilon_2)
+ \epsilon_2\sigma T_2^4
$$

## Python Model

The provided Python function, `planetary_energy_budget`, calculates the surface and atmospheric temperatures, as well as the outgoing longwave radiation based on the given input parameters. The model utilizes a transfer matrix and solves the planetary energy budget equations to determine the temperatures at the surface and in each atmospheric layer.

In [ ]:
# import libraries
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def planetary_energy_budget(solar=1366.0, albedo=0.3, eps1=0.5, eps2=0.5, Hs=50.0, HL=50.0):
    # solar   Solar constant at top of atmosphere (W/m^2)
    # albedo  Shortwave albedo of planet ()
    # eps1    Emissivity of lower atmosphere ()
    # eps2    Emissivity of upper atmosphere ()
    # Hs      Convective heat flux from surface to lower atmosphere (W/m^2)
    # HL      Convective heat flux from lower to upper atmosphere (W/m^2)

    sigma = 5.67e-8  # Stefan-Boltzmann constant (W m^-2 K^-4)

    # Post-albedo solar power to heat the surface
    S = solar / 4 * (1 - albedo)

    # Transfer matrix contains the energy budget coefficients for surface and each layer
    transfer = np.array([[-1., eps1, (1. - eps1) * eps2],
                         [eps1, -2. * eps1, eps1 * eps2],
                         [(1. - eps1) * eps2, eps1 * eps2, -2. * eps2]])

    # Right-hand-side vector in the energy budget equations
    rhs = np.array([(Hs - S) / sigma, (HL - Hs) / sigma, -HL / sigma])

    # Solve planetary energy budget
    # by inverting transfer matrix and multiplying by vector from right-hand-side
    temps4 = np.linalg.solve(transfer, rhs)

    # Extract temperatures by taking the fourth root of each element of the solution vector
    Ts = temps4[0] ** 0.25
    T1 = temps4[1] ** 0.25
    T2 = temps4[2] ** 0.25

    # Compute outgoing longwave radiation (OLR) at top of atmosphere (TOA)
    olr = sigma * (Ts ** 4 * (1 - eps1) * (1 - eps2) + eps1 * T1 ** 4 * (1 - eps2) + eps2 * T2 ** 4)

    # Return all the needed information to the calling function
    return {
        'Tsfc': Ts,
        'Tlower': T1,
        'Tupper': T2,
        'outgoing_longwave': olr,
        'solar': solar,
        'albedo': albedo,
        'solar_absorbed': S,
        'emissivity_lower': eps1,
        'emissivity_upper': eps2,
        'convec_surface': Hs,
        'convec_lower': HL,
        'emission_surface': sigma * Ts ** 4,
        'emission_lower': eps1 * sigma * T1 ** 4,
        'emission_upper': eps2 * sigma * T2 ** 4
    }

In [ ]:
# Example usage:
result = planetary_energy_budget()
result

The goal of this exercise is to explore the sensitivity of the model to variations in emissivity and albedo. By systematically varying these parameters, we aim to observe how changes in emissivity and albedo influence surface and atmospheric temperatures. Feel free to reduce the complexity of the model, by setting $\epsilon_2$ to 1.

### 1. Plotting the atmospheric temperature profile for default values

In [ ]:
# define y axis: atmospheric layers
atm_levels = ["surface", "lower", "upper"]
# extract temperature data from results for each layer into a list
atm_temp = [result['Tsfc'], result['Tlower'], result['Tupper']]

In [ ]:
# create informative label from the information saved in the dictionary
# check out formatted strings: https://www.w3schools.com/python/python_string_formatting.asp
lbl = f"albedo={result['albedo']}, e1={result['emissivity_lower']}, e2={result['emissivity_upper']}"
lbl

In [ ]:
# plot
plt.ylabel('Atmospheric layer')
plt.xlabel('Temperature [K]')
plt.title('Atmospheric temperature profile')

plt.plot(atm_temp, atm_levels,label = lbl)

plt.legend()

**BONUS:** find out how to do dictionary subsetting with comprehension:
https://www.geeksforgeeks.org/get-a-subset-of-dict-in-python/

Describe briefly why it is useful to use dictionaries in python
(https://docs.python.org/3/tutorial/datastructures.html#dictionaries)

### 1. Explore the model sensitivity to changes in albedo
#### What are the effects of higher/lower albedo on surface temperature and atmospheric temperature?

**TODO**: Use different values of albedo and plot the temperature profiles


**BONUS:** Write a function for plotting several profiles!

*(hint: input to the function can be a list of result dictionaries, and you can plot them with a foor loop)*

### 2. Explore the model sensitivity to changes in emissivity
#### What are the effects of higher/lower emissivity on surface temperature and atmospheric temperature?

**TODO**: Use different values of `eps1` and `eps2` (separately) and plot the temperature profiles

This is a very simple model, so don't take the actual number very literally here. The aim of this exercise is just to understand the concepts of modeling, changing the inputs to a model (like albedo, epsilon in this case), and how to analyze its output (here: emission, temperature).

### 3. Questions

**Question:** What do the results show? How are the temperature profile affected by the different parameters?


**Question:** From the lecture, remember how (and where) increased CO2 changes the composition and temperature profile of the atmosphere. In this model, which parameter can best reflect the effect of climate change on the atmospheric temperature profile and why?